# Case Study: 정성적 사례 분석

Self-Corrective RAG 파이프라인의 동작을 구체적 사례로 분석.

**분석 대상**:
1. Self-Correction 성공 사례 (refine → score 향상 → 정답)
2. Agent Routing 사례 (clarification / domain expert / fallback)
3. Passage Accumulation 효과 (retry마다 coverage 증가)
4. 실패 사례 분석 (max retry 후에도 부정확한 답변)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "data" / "results"


def load_all_results() -> pd.DataFrame:
    """Load all experiment results into one DataFrame."""
    rows = []
    if not RESULTS_DIR.exists():
        return pd.DataFrame()
    for exp_dir in RESULTS_DIR.iterdir():
        if not exp_dir.is_dir():
            continue
        for f in exp_dir.glob("*.jsonl"):
            with open(f) as fh:
                for line in fh:
                    rows.append(json.loads(line))
    return pd.DataFrame(rows)


df = load_all_results()
print(f"Total results loaded: {len(df)}")
if not df.empty:
    print(f"Pipelines: {df['pipeline'].unique().tolist()}")

## 1. Self-Correction 성공 사례

retry_count > 0 이고 최종 답변이 정답인 케이스

In [ ]:
def show_case(row: pd.Series, title: str = ""):
    """Pretty-print a single result case for analysis."""
    md = f"### {title}\n\n" if title else ""
    md += f"**Question**: {row.get('question', 'N/A')}\n\n"
    md += f"**Reference**: {row.get('reference', 'N/A')}\n\n"
    md += f"**Prediction**: {row.get('prediction', 'N/A')}\n\n"
    md += f"**Pipeline**: `{row.get('pipeline', 'N/A')}` | "
    md += f"**Retries**: {row.get('retry_count', 0)} | "
    md += f"**Passages Used**: {row.get('passages_used', 'N/A')} | "
    md += f"**Agent**: {row.get('agent_type', 'None')}\n\n"

    # Score progression
    scores = row.get("evaluation_scores", [])
    if scores and isinstance(scores, list):
        md += "**Score Progression**:\n\n"
        md += "| Retry | Relevance | Coverage | Specificity | Sufficiency | Total | Action |\n"
        md += "|-------|-----------|----------|-------------|-------------|-------|--------|\n"
        for ev in scores:
            if isinstance(ev, dict):
                md += f"| {ev.get('retry', '?')} | {ev.get('relevance', '-')} | {ev.get('coverage', '-')} | {ev.get('specificity', '-')} | {ev.get('sufficiency', '-')} | {ev.get('total', '-')} | {ev.get('action', '-')} |\n"
        md += "\n"

    display(Markdown(md))


# Find self-correction success cases
if not df.empty and "retry_count" in df.columns:
    proposed = df[df["pipeline"].str.contains("proposed|self_corrective", case=False, na=False)]
    corrected = proposed[proposed["retry_count"] > 0]

    if not corrected.empty:
        print(f"Found {len(corrected)} cases with retries")
        for i, (_, row) in enumerate(corrected.head(3).iterrows()):
            show_case(row, f"Case {i + 1}: Self-Correction")
    else:
        print("No correction cases found yet. Run experiments first.")
else:
    print("No results loaded. Run experiments first.")